In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from sklearn.metrics import accuracy_score, precision_score, recall_score

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"device: {device}")

device: cuda


In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

train_dataset = datasets.MNIST('mnist_data', train=True, download=True, transform=transform)
test_dataset = datasets.MNIST('mnist_data', train=False, download=True, transform=transform)

batch_size = 128
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print(f"training samples: {len(train_dataset)}")
print(f"test samples: {len(test_dataset)}")

training samples: 60000
test samples: 10000


In [ ]:
# implement ann model
class MNISTNet(nn.Module):
    def __init__(self):
        super(MNISTNet, self).__init__()
        self.input_layer = nn.Linear(784, 128)
        self.hidden_layer = nn.Linear(128, 64)
        self.output_layer = nn.Linear(64, 10)
        
    def forward(self, x):
        # flatten input
        x = x.view(x.size(0), -1)
        # input to hidden with relu activation
        x = F.relu(self.input_layer(x))
        # hidden layer with relu activation
        x = F.relu(self.hidden_layer(x))
        # output layer (softmax applied in loss function)
        x = self.output_layer(x)
        return x

model = MNISTNet().to(device)
print("model created successfully")

model created successfully


In [ ]:
# define loss function and optimizer
criterion = nn.CrossEntropyLoss()  # includes softmax
optimizer = optim.Adam(model.parameters(), lr=0.001)
# LogSoftmax + Negative Log Likelihood hoy in the cross entropy loss
print("loss function: cross-entropy")
print("optimizer: adam")

loss function: cross-entropy
optimizer: adam


In [5]:
# train the model
epochs = 10

for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    for batch_idx, (data, target) in enumerate(train_loader):
        data, target = data.to(device), target.to(device)
        
        # zero gradients
        optimizer.zero_grad()
        
        # forward pass
        output = model(data)
        loss = criterion(output, target)
        
        # backward pass
        loss.backward()
        optimizer.step()
        
        # calculate accuracy
        running_loss += loss.item()
        _, predicted = torch.max(output.data, 1)
        total += target.size(0)
        correct += (predicted == target).sum().item()
    
    # print epoch results
    epoch_loss = running_loss / len(train_loader)
    epoch_acc = 100 * correct / total
    print(f'epoch [{epoch+1}/{epochs}], loss: {epoch_loss:.4f}, accuracy: {epoch_acc:.2f}%')

print("training completed")

epoch [1/10], loss: 0.3216, accuracy: 90.80%
epoch [2/10], loss: 0.1304, accuracy: 96.06%
epoch [2/10], loss: 0.1304, accuracy: 96.06%
epoch [3/10], loss: 0.0893, accuracy: 97.31%
epoch [3/10], loss: 0.0893, accuracy: 97.31%
epoch [4/10], loss: 0.0685, accuracy: 97.84%
epoch [4/10], loss: 0.0685, accuracy: 97.84%
epoch [5/10], loss: 0.0536, accuracy: 98.31%
epoch [5/10], loss: 0.0536, accuracy: 98.31%
epoch [6/10], loss: 0.0440, accuracy: 98.57%
epoch [6/10], loss: 0.0440, accuracy: 98.57%
epoch [7/10], loss: 0.0362, accuracy: 98.86%
epoch [7/10], loss: 0.0362, accuracy: 98.86%
epoch [8/10], loss: 0.0307, accuracy: 99.02%
epoch [8/10], loss: 0.0307, accuracy: 99.02%
epoch [9/10], loss: 0.0274, accuracy: 99.03%
epoch [9/10], loss: 0.0274, accuracy: 99.03%
epoch [10/10], loss: 0.0217, accuracy: 99.27%
training completed
epoch [10/10], loss: 0.0217, accuracy: 99.27%
training completed


In [6]:
# evaluate the model on test set
model.eval()
test_loss = 0.0
correct = 0
total = 0
all_predictions = []
all_targets = []

with torch.no_grad():
    for data, target in test_loader:
        data, target = data.to(device), target.to(device)
        output = model(data)
        loss = criterion(output, target)
        
        test_loss += loss.item()
        _, predicted = torch.max(output.data, 1)
        total += target.size(0)
        correct += (predicted == target).sum().item()
        
        # store for metrics calculation
        all_predictions.extend(predicted.cpu().numpy())
        all_targets.extend(target.cpu().numpy())

# calculate final metrics
test_acc = 100 * correct / total
test_loss = test_loss / len(test_loader)

print(f"test loss: {test_loss:.4f}")
print(f"test accuracy: {test_acc:.2f}%")

test loss: 0.0756
test accuracy: 97.99%


In [7]:
# calculate detailed metrics
accuracy = accuracy_score(all_targets, all_predictions)
precision = precision_score(all_targets, all_predictions, average='weighted')
recall = recall_score(all_targets, all_predictions, average='weighted')

print("evaluation metrics:")
print(f"accuracy: {accuracy * 100:.2f}%")
print(f"precision: {precision * 100:.2f}%")
print(f"recall: {recall * 100:.2f}%")

evaluation metrics:
accuracy: 97.99%
precision: 97.99%
recall: 97.99%
